# State Store

> SQLite-backed workflow state storage for persistence across restarts.

In [ ]:
#| default_exp state_store

In [ ]:
#| export
import json
import sqlite3
from pathlib import Path
from typing import Dict, Any, Optional
from contextlib import contextmanager

## SQLiteWorkflowStateStore

SQLite-backed implementation of workflow state storage. Accepts a plain `session_id: str`
for all operations — no web framework dependency.

Key features:

- State persists across application restarts
- Supports step-specific state storage for progress restoration
- Uses JSON serialization for complex state objects
- Composite key pattern: `flow_id:session_id` for multi-workflow support

In [ ]:
#| export
class SQLiteWorkflowStateStore:
    """SQLite-backed workflow state storage for persistence across restarts."""
    
    def __init__(
        self,
        db_path: Path  # Path to SQLite database file
    ):
        """Initialize the state store and create tables if needed."""
        self.db_path = Path(db_path)
        self.db_path.parent.mkdir(parents=True, exist_ok=True)
        self._init_db()
    
    @contextmanager
    def _get_connection(self):  # Context manager yielding SQLite connection
        """Get a database connection with proper cleanup."""
        conn = sqlite3.connect(self.db_path)
        conn.row_factory = sqlite3.Row
        try:
            yield conn
            conn.commit()
        finally:
            conn.close()
    
    def _init_db(self) -> None:
        """Initialize database schema."""
        with self._get_connection() as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS workflow_state (
                    flow_session_key TEXT PRIMARY KEY,
                    current_step TEXT,
                    state_json TEXT DEFAULT '{}',
                    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                )
            """)
            conn.execute("""
                CREATE INDEX IF NOT EXISTS idx_workflow_state_updated
                ON workflow_state(updated_at)
            """)
    
    def _make_key(
        self,
        flow_id: str,  # Workflow identifier
        session_id: str  # Session identifier string
    ) -> str:  # Composite key for storage
        """Create composite key from flow_id and session_id."""
        return f"{flow_id}:{session_id}"
    
    def get_current_step(
        self,
        flow_id: str,  # Workflow identifier
        session_id: str  # Session identifier string
    ) -> Optional[str]:  # Current step ID or None
        """Get current step ID for a workflow."""
        key = self._make_key(flow_id, session_id)
        with self._get_connection() as conn:
            row = conn.execute(
                "SELECT current_step FROM workflow_state WHERE flow_session_key = ?",
                (key,)
            ).fetchone()
            return row["current_step"] if row else None
    
    def set_current_step(
        self,
        flow_id: str,  # Workflow identifier
        session_id: str,  # Session identifier string
        step_id: str  # Step ID to set as current
    ) -> None:
        """Set current step ID for a workflow."""
        key = self._make_key(flow_id, session_id)
        with self._get_connection() as conn:
            conn.execute("""
                INSERT INTO workflow_state (flow_session_key, current_step)
                VALUES (?, ?)
                ON CONFLICT(flow_session_key) DO UPDATE SET
                    current_step = excluded.current_step,
                    updated_at = CURRENT_TIMESTAMP
            """, (key, step_id))
    
    def get_state(
        self,
        flow_id: str,  # Workflow identifier
        session_id: str  # Session identifier string
    ) -> Dict[str, Any]:  # Workflow state dictionary
        """Get all workflow state."""
        key = self._make_key(flow_id, session_id)
        with self._get_connection() as conn:
            row = conn.execute(
                "SELECT state_json FROM workflow_state WHERE flow_session_key = ?",
                (key,)
            ).fetchone()
            if row and row["state_json"]:
                return json.loads(row["state_json"])
            return {}
    
    def update_state(
        self,
        flow_id: str,  # Workflow identifier
        session_id: str,  # Session identifier string
        updates: Dict[str, Any]  # State updates to merge
    ) -> None:
        """Update workflow state by merging new values."""
        key = self._make_key(flow_id, session_id)
        current_state = self.get_state(flow_id, session_id)
        current_state.update(updates)
        state_json = json.dumps(current_state)
        with self._get_connection() as conn:
            conn.execute("""
                INSERT INTO workflow_state (flow_session_key, state_json)
                VALUES (?, ?)
                ON CONFLICT(flow_session_key) DO UPDATE SET
                    state_json = excluded.state_json,
                    updated_at = CURRENT_TIMESTAMP
            """, (key, state_json))
    
    def clear_state(
        self,
        flow_id: str,  # Workflow identifier
        session_id: str  # Session identifier string
    ) -> None:
        """Clear all workflow state."""
        key = self._make_key(flow_id, session_id)
        with self._get_connection() as conn:
            conn.execute(
                "DELETE FROM workflow_state WHERE flow_session_key = ?",
                (key,)
            )
    
    def get_step_state(
        self,
        flow_id: str,  # Workflow identifier
        session_id: str,  # Session identifier string
        step_id: str  # Step identifier
    ) -> Dict[str, Any]:  # Step-specific state dictionary
        """Get state for a specific step."""
        state = self.get_state(flow_id, session_id)
        step_key = f"_step_{step_id}"
        return state.get(step_key, {})
    
    def update_step_state(
        self,
        flow_id: str,  # Workflow identifier
        session_id: str,  # Session identifier string
        step_id: str,  # Step identifier
        updates: Dict[str, Any]  # Step-specific state updates
    ) -> None:
        """Update state for a specific step."""
        step_key = f"_step_{step_id}"
        current_step_state = self.get_step_state(flow_id, session_id, step_id)
        current_step_state.update(updates)
        self.update_state(flow_id, session_id, {step_key: current_step_state})
    
    def clear_step_state(
        self,
        flow_id: str,  # Workflow identifier
        session_id: str,  # Session identifier string
        step_id: str  # Step identifier
    ) -> None:
        """Clear state for a specific step."""
        step_key = f"_step_{step_id}"
        state = self.get_state(flow_id, session_id)
        if step_key in state:
            del state[step_key]
            key = self._make_key(flow_id, session_id)
            state_json = json.dumps(state)
            with self._get_connection() as conn:
                conn.execute("""
                    UPDATE workflow_state SET state_json = ?, updated_at = CURRENT_TIMESTAMP
                    WHERE flow_session_key = ?
                """, (state_json, key))

## Tests

In [ ]:
import tempfile, os

# Create a temporary database for testing
_tmp = tempfile.NamedTemporaryFile(suffix='.db', delete=False)
_tmp_path = _tmp.name
_tmp.close()
store = SQLiteWorkflowStateStore(_tmp_path)

In [ ]:
# Step tracking
assert store.get_current_step("flow1", "sess1") is None

store.set_current_step("flow1", "sess1", "selection")
assert store.get_current_step("flow1", "sess1") == "selection"

store.set_current_step("flow1", "sess1", "decomposition")
assert store.get_current_step("flow1", "sess1") == "decomposition"

# Different sessions are independent
assert store.get_current_step("flow1", "sess2") is None

print("Step tracking tests passed!")

Step tracking tests passed!


In [ ]:
# State CRUD
assert store.get_state("flow1", "sess1") == {}

store.update_state("flow1", "sess1", {"count": 5, "name": "test"})
state = store.get_state("flow1", "sess1")
assert state["count"] == 5
assert state["name"] == "test"

# Merge semantics
store.update_state("flow1", "sess1", {"count": 10})
state = store.get_state("flow1", "sess1")
assert state["count"] == 10
assert state["name"] == "test"  # preserved

# Clear
store.clear_state("flow1", "sess1")
assert store.get_state("flow1", "sess1") == {}

print("State CRUD tests passed!")

State CRUD tests passed!


In [ ]:
# Step-specific state
assert store.get_step_state("flow1", "sess1", "selection") == {}

store.update_step_state("flow1", "sess1", "selection", {"sources": ["a", "b"]})
step_state = store.get_step_state("flow1", "sess1", "selection")
assert step_state["sources"] == ["a", "b"]

# Different steps are independent
store.update_step_state("flow1", "sess1", "decomposition", {"segments": [1, 2, 3]})
assert store.get_step_state("flow1", "sess1", "selection")["sources"] == ["a", "b"]
assert store.get_step_state("flow1", "sess1", "decomposition")["segments"] == [1, 2, 3]

# Clear step state
store.clear_step_state("flow1", "sess1", "selection")
assert store.get_step_state("flow1", "sess1", "selection") == {}
assert store.get_step_state("flow1", "sess1", "decomposition")["segments"] == [1, 2, 3]  # preserved

print("Step-specific state tests passed!")

Step-specific state tests passed!


In [ ]:
# Flow and session isolation
store.clear_state("flow1", "sess1")
store.update_state("flow1", "sess1", {"flow": 1})
store.update_state("flow2", "sess1", {"flow": 2})
store.update_state("flow1", "sess2", {"flow": 1, "session": 2})

assert store.get_state("flow1", "sess1")["flow"] == 1
assert store.get_state("flow2", "sess1")["flow"] == 2
assert store.get_state("flow1", "sess2")["session"] == 2

print("Isolation tests passed!")

Isolation tests passed!


In [ ]:
# Cleanup
os.unlink(_tmp_path)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()